# GPT 시리즈와 스케일링 법칙 실습 - Tiny Scaling Law Fitting

- Tutorial ID: `mod-gpt-scaling-lab`
- Tutorial: GPT 시리즈와 스케일링 법칙 실습
- Section ID: `gpt-1`
- Section: Tiny Scaling Law Fitting


In [ ]:
# ============================================================
# 코드 읽는 법 — Tiny Scaling Law Fitting
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
N = np.array([1e6, 3e6, 1e7, 3e7, 1e8])
true_loss = 1.55 + 2.2 * N**(-0.095)
noise = np.array([0.015,-0.006,0.01,-0.004,0.002])
loss = true_loss + noise
# irreducible loss L_inf를 가정하고 log(loss-L_inf)=a - alpha log N
L_inf = 1.55
x = np.log(N); y = np.log(loss - L_inf)
alpha = -np.polyfit(x, y, 1)[0]
print('model sizes:', N.astype(int))
print('loss:', loss)
print('estimated scaling exponent alpha =', round(alpha, 3))
next_N = 1e9
pred = L_inf + np.exp(np.polyfit(x, y, 1)[1]) * next_N**(-alpha)
print('predicted loss at 1B params:', round(pred, 4))